In [1]:
import pandas as pd
df = pd.read_csv('/Users/cyruskurd/Documents/grad_programming/AML/Project work/Deliverable 3/combined_data_with_y_ta.csv')

In [3]:
df.isna().sum()

timestamp             0
open                  0
high                  0
low                   0
close                 0
vol                   0
amount                0
year                  0
month                 0
day                   0
ticker                0
y                     0
SMA_10            48132
SMA_20           101612
EMA_10            48132
EMA_20           101612
MACD_12_26_9     133700
MACDh_12_26_9    176484
MACDs_12_26_9    176484
RSI_14            74872
BBL_20_2.0       101612
BBM_20_2.0       101612
BBU_20_2.0       101612
BBB_20_2.0       101612
BBP_20_2.0       101612
ATR_14            74872
OBV                   0
dtype: int64

In [6]:
df = df.dropna()

In [7]:
df

,timestamp,open,high,low,close,vol,amount,year,month,day,...,MACDh_12_26_9,MACDs_12_26_9,RSI_14,BBL_20_2.0,BBM_20_2.0,BBU_20_2.0,BBB_20_2.0,BBP_20_2.0,ATR_14,OBV
33,2000-03-03,12.36,13.55,12.35,13.48,9226800.0,1.219935e+08,2000.0,3.0,3.0,...,-0.067962,0.788294,68.016260,10.518610,12.0695,13.620390,25.699321,0.954739,0.811840,28822523.0
34,2000-03-06,13.55,13.65,12.50,12.77,5583700.0,7.311043e+07,2000.0,3.0,6.0,...,-0.069366,0.770953,59.308383,10.620636,12.1475,13.674364,25.138744,0.703849,0.838109,23238823.0
35,2000-03-07,12.65,13.00,12.37,12.80,1489300.0,1.896957e+07,2000.0,3.0,7.0,...,-0.071724,0.753022,59.544067,10.740451,12.2275,13.714549,24.323027,0.692495,0.822043,24728123.0
36,2000-03-08,12.80,13.27,12.64,13.26,2381700.0,3.096094e+07,2000.0,3.0,8.0,...,-0.046951,0.741284,63.075569,10.979877,12.3555,13.731123,22.267381,0.828760,0.807303,27109823.0
37,2000-03-09,13.26,13.70,12.66,13.11,3005018.0,3.934368e+07,2000.0,3.0,9.0,...,-0.045293,0.729961,61.199523,11.303415,12.4775,13.651585,18.819231,0.769359,0.825069,24104805.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14989875,2024-09-23,47.15,47.69,46.80,46.98,872500.0,4.126426e+07,2024.0,9.0,23.0,...,0.007846,-3.061110,24.076741,45.885458,50.8345,55.783542,19.471195,0.110581,1.685181,-43082900.0
14989876,2024-09-24,47.20,49.10,46.41,48.95,1734200.0,8.336652e+07,2024.0,9.0,24.0,...,0.170056,-3.018596,35.877604,45.724579,50.6300,55.535421,19.377529,0.328761,1.757686,-41348700.0
14989877,2024-09-25,48.99,50.56,48.96,49.24,1644100.0,8.187121e+07,2024.0,9.0,25.0,...,0.308839,-2.941386,37.419637,45.632187,50.4375,55.242813,19.054526,0.375398,1.747037,-39704600.0
14989878,2024-09-26,49.20,50.78,48.88,50.76,1352200.0,6.758263e+07,2024.0,9.0,26.0,...,0.503455,-2.815522,44.899134,45.734668,50.2830,54.831332,18.090933,0.552437,1.758059,-38352400.0


In [8]:
df.to_csv('/Users/cyruskurd/Documents/grad_programming/AML/Project work/Deliverable 3/combined_data_with_y_ta.csv', index=False)

In [2]:
import pandas as pd
# Load and sort data
df = pd.read_csv('/Users/cyruskurd/Documents/grad_programming/AML/Project work/Deliverable 3/combined_data_with_y_ta.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
df.dropna(inplace=True)


In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from xgboost import XGBClassifier


# Define features and target
X = df.copy().drop(columns='y')
y = df['y']

# Drop non-numeric columns (e.g., timestamp) before scaling
non_numeric_cols = ['timestamp']  # Adjust column names based on your actual data
X_numeric = X.drop(columns=non_numeric_cols)

# Feature Scaling
scaler = StandardScaler()
X_scaled_numeric = scaler.fit_transform(X_numeric)

# Replace the original X_scaled with the scaled numeric features
X_scaled = X_numeric.copy()
X_scaled.loc[:, :] = X_scaled_numeric

# Train-test-validation split
split_date = '2020-01-01'

# Convert y to binary labels based on a threshold
binary_threshold = 0.1
y_binary = (y >= binary_threshold).astype(int)

# Ensure timestamp column is used only for filtering, not included in training data
train_mask = df['timestamp'] < split_date
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_scaled[train_mask],
    y_binary[train_mask],
    test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)

# Handle class imbalance using scale_pos_weight
negative_counts = (y_train == 0).sum()
positive_counts = (y_train == 1).sum()
scale_pos_weight = negative_counts / positive_counts

# Train XGBoost classifier
xgb_model = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, n_estimators=100,
                          max_depth=5, learning_rate=0.1, subsample=0.8)

xgb_model.fit(X_train, y_train)

# Predictions
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
threshold = 0.5
y_pred = (y_pred_proba >= threshold).astype(int)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"XGBoost Accuracy: {accuracy:.4f}")
print(f"XGBoost F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")
print("Confusion Matrix:\n", conf_matrix)
print(classification_report(y_test, y_pred))

/var/folders/3n/p64dgb2j7p1bswd6n4tr8m4w0000gn/T/ipykernel_2126/2018611826.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 1.01054571 -1.14780385  1.01030089 ...  1.02295585  1.01200742
 -0.06443475]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_scaled.loc[:, :] = X_scaled_numeric


XGBoost Accuracy: 0.9386
XGBoost F1 Score: 0.4557
ROC AUC Score: 0.9635
Confusion Matrix:
 [[1746991  110690]
 [   6898   49217]]
              precision    recall  f1-score   support

           0       1.00      0.94      0.97   1857681
           1       0.31      0.88      0.46     56115

    accuracy                           0.94   1913796
   macro avg       0.65      0.91      0.71   1913796
weighted avg       0.98      0.94      0.95   1913796



In [15]:
import numpy as np
import pandas as pd
import backtrader as bt
from backtrader.feeds import PandasData
from tqdm import tqdm

class SignalData(PandasData):
    lines = ('predicted',)
    params = (('predicted', -1),)

class MLStrategy(bt.Strategy):
    params = dict(
        n_positions=10,
        verbose=False,
    )
    
    def __init__(self):
        self.order = None
    
    def next(self):
        if self.order:
            return
        
        # Get predictions from data
        preds = [(d._name, d.predicted[0]) for d in self.datas]
        preds = sorted(preds, key=lambda x: x[1], reverse=True)
        
        # Take top predictions
        top_preds = preds[:self.p.n_positions]
        
        # Place buy orders for top predictions
        for name, pred in top_preds:
            data = self.getdatabyname(name)
            size = self.broker.getcash() / len(top_preds) / data.close[0]
            self.order = self.buy(data=data, size=size)

# Backtrader setup
cerebro = bt.Cerebro()
cerebro.addstrategy(MLStrategy)

# Add data to Backtrader
for ticker in df['ticker'].unique():
    ticker_data = df[df['ticker'] == ticker].copy()
    ticker_data.index = pd.to_datetime(ticker_data['timestamp'])
    ticker_data = ticker_data[['open', 'high', 'low', 'close', 'vol', 'predicted']]
    
    # Ensure data is sorted
    ticker_data = ticker_data.sort_index()
    
    data_feed = SignalData(dataname=ticker_data)
    cerebro.adddata(data_feed, name=ticker)

# Set broker cash
cerebro.broker.setcash(100000)

# Run backtesting
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())
results = cerebro.run()
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

# Plot results
cerebro.plot()

Starting Portfolio Value: 100000.00


IndexError: invalid index to scalar variable.